Import packages

In [93]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import cobra
from cobra.io import read_sbml_model, write_sbml_model
from cobra.flux_analysis import flux_variability_analysis
from tqdm import tqdm

In [109]:
from pathlib import Path
import plotly
import plotly.express as px
import scipy.stats
os.environ["OMP_NUM_THREADS"] = '1' # because of a data leak of KMeans on windows
import fba_comparison_stats as cmp
from efflux_method import *
import networkx as nx
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "browser"

Import the models

How to make each models, how to put information in it? Efflux?

In [95]:
M_xanthus = read_sbml_model("../M_xanthus_model_V3_hdca.xml")

In [96]:
list_of_genes = []
for i in M_xanthus.genes:
    list_of_genes.append(i.id)

In [97]:
# Getting fluxes with constrains
dico = {14: "Alone", 2: "Predation"}
for i in [14, 2]:
    M_xanthus = read_sbml_model("../M_xanthus_model_V3_hdca.xml")
    DictAP = read_csv_data(
        "/home/mickael/github/M_xanthus-E_coli-Predation/data/Raw/WT_vs_4preys_iMAT.csv",
        id_gene=1,
        id_val=i,
        list_of_genes = list_of_genes,
        head=True,
        quantile=0.95,
    )
    Eflux(M_xanthus, DictAP, const=100, default_exp_val=1, ignore_human=True)
    M_xanthus.reactions.EX_glc_D_e.bounds = (0, 1000)
    model_c = M_xanthus.copy()
    write_sbml_model(
        model_c,
        "/home/mickael/github/M_xanthus-E_coli-Predation/fba_comparer-main/Models_Preda/M_xanthus_V3_hdca_efflux_"
        + dico[i]
        + "_C"
        + ".xml",
    )
print(dico[i] + " Done!")

Predation Done!


In [98]:
M_xanthus_alone = read_sbml_model("Models_Preda/M_xanthus_V3_hdca_efflux_Alone_C.xml")
solution_alone = M_xanthus_alone.optimize()

In [99]:
M_xanthus_predation = read_sbml_model("Models_Preda/M_xanthus_V3_hdca_efflux_Predation_C.xml")
solution_predation = M_xanthus_predation.optimize()

In [100]:
print(f"Alone:\t{solution_alone.objective_value}")
print(f"Predation:\t{solution_predation.objective_value}")

Alone:	0.204452758536134
Predation:	0.10165517381423969


In [101]:
cmp.help()

Available functions:
- build_dataframe(model, solutions, conditions)
- scatter_plot_fluxes(fba, conditions)         
- correlation_matrix(fba, conditions, [opt: color])
- filter_dataframe(fba, conditions)
- bar_plot_flux_variation(fba, [opt: start, end])         
- hierarchical_clustering_std_dev(fba, [opt: start, end, d_max])
- pca(fba, base_condition, other_conditions, [opt: clusters, dimensions])         
- normalize_dataframe_rows(fba, conditions [opt: method])
- normalize_dataframe_cols(fba, conditions, obj_values)         
- pca_color_pathways(fba, base_condition, other_conditions, pathway, pathway_dict, [opt: dimensions=2])         
- summarize_pathways(id_list, id_dict)
- flux_coupling_matrix(fba, conditions, remove_unchanging = 0.01, abs_values = True)        
- flux_coupling_network(flux_coupling_matrix = pd.DataFrame(), pc = 0.8)
- color_network_by_pathway(nw, pathway, pathway_dict)        
- color_network_by_communities(nw, iterations)
- visualize_network(nw, [opt: layout])

## **FBA Comparer**
**Create combined dataframe and preprocess data**

Filter all reactions where fluxes are zero:

In [102]:
solutions = [solution_alone, solution_predation] 
conditions = ["Alone", "Predation"]
obj_values = [solution_alone.objective_value, solution_predation.objective_value]

mxanthus = cmp.build_dataframe(M_xanthus_alone, solutions, conditions)
mxanthus_filtered = cmp.filter_dataframe(mxanthus, conditions, rounding = True)
print(f"Number of reactions in the models: {len(mxanthus)}")
print(f"Number of nonzero reactions in the models: {len(mxanthus_filtered)}")

Number of reactions in the models: 1339
Number of nonzero reactions in the models: 449


normalize data:

In [103]:
mxanthus_normalized = cmp.normalize_dataframe_cols(mxanthus_filtered, conditions, obj_values) # biomass normalization
mxanthus_normalized_div_by_max = cmp.normalize_dataframe_rows(mxanthus_normalized, conditions, "div_by_max") # normalization reactions
# note: there is no difference between the order of div_by_max normalization and filtering for changing reactions
mxanthus_normalized_div_by_max_changing = mxanthus_normalized_div_by_max[mxanthus_normalized_div_by_max['Std_dev'] >= 0.01] # keep only reactions that change
print(f"Number of normalized changing reactions: {len(mxanthus_normalized_div_by_max_changing)}")

Number of normalized changing reactions: 157


load additional data:

In [104]:
# load KEGG pathway dictionary (maps pathways to reactions)
pathway_dict = {}
with open('pathway_information/ecoli_map_pathway_to_id.tsv', 'r') as file:
    for line in file:
        words = line.strip().replace("_", " ").split("\t")
        pathway_dict[words.pop(0)] = words

## load ID dictionary (mapping ids to pathways) 
filtered_ids = []
with open('pathway_information/ecoli_filtered_ids.tsv', 'r') as file:
    for line in file:
        filtered_ids.append(line.strip())
id_dict = {}
with open('pathway_information/ecoli_map_id_to_pathway.tsv', 'r') as file:
    for id, line in zip(filtered_ids, file):
        if line == "\n":
            continue
        words = line.strip().split("\t")
        id_dict[id] = words

# load KEGG pathway hierarchy
pathway_hierarchy = {}
with open('pathway_information/kegg_pathway_hierarchy.txt', 'r') as file:
    for line in file:
        line = line.rstrip('\n')
        words = line.replace("_", " ").split("\\t")
        pathway_hierarchy[words.pop(0)] = words

## Identify reactions with most/least variation

In [ ]:
#scatter plot
fig = cmp.scatter_plot_fluxes(mxanthus_normalized, conditions)
fig

Most affected / change reaction

In [112]:
most_variable_bar_plot = cmp.bar_plot_flux_variation(mxanthus_normalized, conditions, 0, 20)
most_variable_bar_plot.show()

In [105]:
sorted_flux = mxanthus_normalized[["ID", "Name", "Std_dev"]].sort_values(by=['Std_dev'], ascending=False) 
sorted_flux["Kegg"] = [M_xanthus_alone.reactions.get_by_id(id).__dict__["_annotation"].get("kegg.reaction") for id in sorted_flux["ID"]]
sorted_flux[0:20]

,ID,Name,Std_dev,Kegg
583,rxn01508_c,ATP:deoxyadenosine 5'-phosphotransferase [c],7.901103,R02089
842,rxn01507_c,2'-Deoxyadenosine 5'-monophosphate phosphohydr...,7.901103,R02088
672,rxn05312_c,Inorganic phosphate transporter [c],4.471832,None
1261,EX_pi_e,Exchange for Phosphate [e],4.471832,None
342,rxn01444_c,ATP:deoxyguanosine 5'-phosphotransferase [c],3.859427,R01967
910,rxn00709_c,ATP:uridine 5'-phosphotransferase [c],3.462693,R00964
716,rxn01366_c,Uridine:phosphate alpha-D-ribosyltransferase [c],3.462693,R01876
190,rxn00913_c,Guanosine 5'-monophosphate phosphohydrolase [c],3.272627,R01227
450,rxn01548_c,guanosine:phosphate alpha-D-ribosyltransferase...,3.272627,R02147
352,rxn00463_c,Uridine triphosphate pyrophosphohydrolase [c],3.071631,R00662


Flux correlation network

In [106]:
corr_matrix = cmp.flux_coupling_matrix(mxanthus_normalized_div_by_max, conditions, remove_unchanging=0.01, abs_values=True)

292 reaction were removed, because their Std_dev is lower than 0.01
The following 1 reactions were removed because they swap signs across conditions: ['rxn01200_c']


In [ ]:
nw = cmp.flux_coupling_network(corr_matrix, pc=0.8)
fig = cmp.visualize_interactive_network(nw, mxanthus_normalized_div_by_max, conditions)
fig.show()

Created Flux Coupling network with pc threshold 0.8.
Number of nodes: 156
Number of edges: 6042
Number of connected components: 2
